# 🛰️ TerraSight — Satellite Image Change Detection
### Siamese Network with Multi-Level Features + Hybrid Focal-Dice Loss
---
**VSCode / Local Environment Edition** — all Colab-specific code removed

**Run cells in order from top to bottom. Do not skip any cell.**

## CELL 1 — Install Dependencies
Run this once. After it finishes, **restart the kernel**, then run all remaining cells.

In [ ]:
import subprocess, sys

packages = [
    'torch torchvision',
    'opencv-python==4.9.0.80',
    'albumentations==1.3.1',
    'scikit-learn',
    'matplotlib',
    'seaborn',
    'tqdm',
]

for pkg in packages:
    print(f'Installing {pkg}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkg.split())

print('\n✅ All packages installed! Now RESTART THE KERNEL, then run from Cell 2.')

## CELL 2 — Imports & Device Check

In [ ]:
import torch
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device    : {DEVICE}')
print('\n✅ Imports OK!')

## CELL 3 — Configuration
**Edit `BASE_DIR` to match your actual project folder path.**

In [ ]:
import os

# ── Paths ──────────────────────────────────────────────────────────────
# Change this to your actual TerraSight folder
BASE_DIR       = r'C:\Users\apraj\OneDrive\Desktop\TerraSight'
DATA_DIR       = os.path.join(BASE_DIR, 'data')
RAW_DIR        = os.path.join(DATA_DIR, 'raw')
PROCESSED_DIR  = os.path.join(DATA_DIR, 'processed')
CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints')
RESULTS_DIR    = os.path.join(BASE_DIR, 'results')

for d in [BASE_DIR, DATA_DIR, RAW_DIR, PROCESSED_DIR, CHECKPOINT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Dataset ────────────────────────────────────────────────────────────
PATCH_SIZE       = 80
PATCH_STRIDE     = 40
CHANGE_THRESHOLD = 0.1

# ── Model ──────────────────────────────────────────────────────────────
BACKBONE        = 'vgg16'   # 'vgg16' or 'resnet50'
USE_BLOCKS      = [4, 5]
PRETRAINED      = True
FREEZE_BACKBONE = True

# ── Training ───────────────────────────────────────────────────────────
BATCH_SIZE    = 16          # reduced from 32 for local CPU — increase if you have GPU
EPOCHS        = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 1e-5
LR_STEP_SIZE  = 10
LR_GAMMA      = 0.5

# ── Loss ───────────────────────────────────────────────────────────────
FOCAL_ALPHA  = 0.25
FOCAL_GAMMA  = 2.0
FOCAL_WEIGHT = 0.5
DICE_WEIGHT  = 0.5

print('✅ Configuration loaded')
print(f'   Base dir     : {BASE_DIR}')
print(f'   Patch size   : {PATCH_SIZE}x{PATCH_SIZE}')
print(f'   Backbone     : {BACKBONE}')
print(f'   Batch size   : {BATCH_SIZE}')
print(f'   Epochs       : {EPOCHS}')
print(f'   Device       : {DEVICE}')

## CELL 4 — Create Synthetic Demo Dataset
This creates a local synthetic dataset so you can run the full pipeline immediately.
Replace with real LEVIR-CD data when available.

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

print('Creating synthetic demo dataset...')
print('(This lets you test the full pipeline now)')
print()

def create_demo_dataset(raw_dir, n_train=300, n_val=80, n_test=80):
    splits = {'train': n_train, 'val': n_val, 'test': n_test}

    for split, n in splits.items():
        for folder in ['A', 'B', 'label']:
            os.makedirs(os.path.join(raw_dir, split, folder), exist_ok=True)

        for i in tqdm(range(n), desc=f'Creating {split}'):
            base = np.random.randint(60, 180, (256, 256, 3), dtype=np.uint8)
            for _ in range(np.random.randint(3, 8)):
                x1 = np.random.randint(0, 200)
                y1 = np.random.randint(0, 200)
                x2 = x1 + np.random.randint(20, 60)
                y2 = y1 + np.random.randint(20, 60)
                color = [int(c) for c in np.random.randint(80, 200, 3)]
                cv2.rectangle(base, (x1, y1), (x2, y2), color, -1)

            img_a = base.copy()
            img_b = base.copy()
            mask  = np.zeros((256, 256), dtype=np.uint8)

            if np.random.random() > 0.5:
                cx = np.random.randint(30, 200)
                cy = np.random.randint(30, 200)
                w  = np.random.randint(20, 50)
                h  = np.random.randint(20, 50)
                building_color = [int(c) for c in np.random.randint(150, 250, 3)]
                cv2.rectangle(img_b, (cx, cy), (cx+w, cy+h), building_color, -1)
                cv2.rectangle(mask,  (cx, cy), (cx+w, cy+h), 255, -1)

            noise = np.random.randint(-15, 15, img_b.shape, dtype=np.int16)
            img_b = np.clip(img_b.astype(np.int16) + noise, 0, 255).astype(np.uint8)

            fname = f'{i:05d}.png'
            cv2.imwrite(os.path.join(raw_dir, split, 'A', fname), img_a)
            cv2.imwrite(os.path.join(raw_dir, split, 'B', fname), img_b)
            cv2.imwrite(os.path.join(raw_dir, split, 'label', fname), mask)

        print(f'  {split}: {n} image pairs created')

    print()
    print('✅ Demo dataset created!')
    print(f'   Location: {raw_dir}')


create_demo_dataset(RAW_DIR, n_train=300, n_val=80, n_test=80)

print()
print('=== DATASET STRUCTURE ===')
for split in ['train', 'val', 'test']:
    a_dir = os.path.join(RAW_DIR, split, 'A')
    if os.path.exists(a_dir):
        n = len(os.listdir(a_dir))
        print(f'  {split:6s}/A/     : {n} images')
        print(f'  {split:6s}/B/     : {n} images')
        print(f'  {split:6s}/label/ : {n} masks')

## CELL 5 — Explore Dataset

In [ ]:
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')   # use non-interactive backend for VSCode
import matplotlib.pyplot as plt
import os

def explore_dataset(data_dir):
    splits = ['train', 'val', 'test']
    print('=== DATASET OVERVIEW ===')
    for split in splits:
        a_dir = os.path.join(data_dir, split, 'A')
        if os.path.exists(a_dir):
            n = len(os.listdir(a_dir))
            print(f'  {split:6s}: {n} image pairs')

    train_a = os.path.join(data_dir, 'train', 'A')
    train_b = os.path.join(data_dir, 'train', 'B')
    train_m = os.path.join(data_dir, 'train', 'label')

    if not os.path.exists(train_a):
        print('\nDataset not found. Please run Cell 4 first.')
        return

    samples = os.listdir(train_a)[:4]
    fig, axes = plt.subplots(4, 3, figsize=(12, 16))
    fig.suptitle('Dataset — Sample Image Pairs', fontsize=14)

    total_changed = 0
    total_pixels  = 0

    for idx, name in enumerate(samples):
        img_a = cv2.cvtColor(cv2.imread(os.path.join(train_a, name)), cv2.COLOR_BGR2RGB)
        img_b = cv2.cvtColor(cv2.imread(os.path.join(train_b, name)), cv2.COLOR_BGR2RGB)
        mask  = cv2.imread(os.path.join(train_m, name), 0)
        mask_bin = (mask > 127).astype(np.uint8)

        total_changed += mask_bin.sum()
        total_pixels  += mask_bin.size
        change_pct     = mask_bin.mean() * 100

        axes[idx,0].imshow(img_a); axes[idx,0].set_title('Before'); axes[idx,0].axis('off')
        axes[idx,1].imshow(img_b); axes[idx,1].set_title('After');  axes[idx,1].axis('off')
        axes[idx,2].imshow(mask_bin, cmap='RdYlGn_r', vmin=0, vmax=1)
        axes[idx,2].set_title(f'Change Mask\n{change_pct:.1f}% changed'); axes[idx,2].axis('off')

    plt.tight_layout()
    out_path = os.path.join(RESULTS_DIR, 'dataset_samples.png')
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Plot saved to {out_path}')

    print(f'\n=== CLASS IMBALANCE ANALYSIS ===')
    print(f'  Changed   : {total_changed:,} pixels ({total_changed/total_pixels:.1%})')
    print(f'  Unchanged : {total_pixels-total_changed:,} pixels ({1-total_changed/total_pixels:.1%})')
    print(f'  Imbalance : 1:{(total_pixels-total_changed)//max(total_changed,1)}')
    print('  → This is WHY we need Focal + Dice loss!')

explore_dataset(RAW_DIR)

## CELL 6 — Patch Extraction

In [ ]:
import cv2
import numpy as np
import os
from tqdm import tqdm

class PatchExtractor:
    def __init__(self, patch_size=80, stride=40, threshold=0.1):
        self.patch_size = patch_size
        self.stride     = stride
        self.threshold  = threshold

    def extract_from_image_pair(self, img_a, img_b, mask):
        patches = []
        h, w = img_a.shape[:2]
        for y in range(0, h - self.patch_size + 1, self.stride):
            for x in range(0, w - self.patch_size + 1, self.stride):
                pa = img_a[y:y+self.patch_size, x:x+self.patch_size]
                pb = img_b[y:y+self.patch_size, x:x+self.patch_size]
                pm = mask[y:y+self.patch_size,  x:x+self.patch_size]
                ratio = (pm > 127).mean()
                label = 1 if ratio > self.threshold else 0
                patches.append((pa, pb, pm, label))
        return patches

    def process_split(self, raw_dir, out_dir, split):
        a_dir = os.path.join(raw_dir, split, 'A')
        b_dir = os.path.join(raw_dir, split, 'B')
        m_dir = os.path.join(raw_dir, split, 'label')

        if not os.path.exists(a_dir):
            print(f'Skipping {split} — not found at {a_dir}')
            return 0

        out_a = os.path.join(out_dir, split, 'A')
        out_b = os.path.join(out_dir, split, 'B')
        out_m = os.path.join(out_dir, split, 'mask')
        for d in [out_a, out_b, out_m]:
            os.makedirs(d, exist_ok=True)

        labels      = []
        idx         = 0
        n_change    = 0
        n_no_change = 0
        img_names   = [f for f in os.listdir(a_dir) if f.endswith(('.png', '.jpg', '.tif'))]

        for name in tqdm(img_names, desc=f'Processing {split}'):
            ia = cv2.imread(os.path.join(a_dir, name))
            ib = cv2.imread(os.path.join(b_dir, name))
            im = cv2.imread(os.path.join(m_dir, name), 0)
            if ia is None or ib is None or im is None:
                continue
            for pa, pb, pm, label in self.extract_from_image_pair(ia, ib, im):
                fname = f'patch_{idx:07d}.png'
                cv2.imwrite(os.path.join(out_a, fname), pa)
                cv2.imwrite(os.path.join(out_b, fname), pb)
                cv2.imwrite(os.path.join(out_m, fname), pm)
                labels.append(f'{fname},{label}\n')
                if label == 1: n_change    += 1
                else:          n_no_change += 1
                idx += 1

        with open(os.path.join(out_dir, split, 'labels.csv'), 'w') as f:
            f.writelines(labels)

        print(f'  {split}: {idx} patches | Changed: {n_change} | Unchanged: {n_no_change}')
        return idx


extractor = PatchExtractor(patch_size=PATCH_SIZE, stride=PATCH_STRIDE, threshold=CHANGE_THRESHOLD)

print('Extracting patches...')
for split in ['train', 'val', 'test']:
    extractor.process_split(RAW_DIR, PROCESSED_DIR, split)

print('\n✅ Patch extraction complete!')

## CELL 7 — Dataset & DataLoaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import cv2, os
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2


def get_transforms(split='train'):
    if split == 'train':
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.3),
            A.GaussNoise(p=0.2),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ], additional_targets={'image_b': 'image'})
    else:
        return A.Compose([
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ], additional_targets={'image_b': 'image'})


class ChangeDetectionDataset(Dataset):
    def __init__(self, data_dir, split='train'):
        self.root      = os.path.join(data_dir, split)
        self.transform = get_transforms(split)
        self.samples   = []
        label_file = os.path.join(self.root, 'labels.csv')
        if not os.path.exists(label_file):
            print(f'labels.csv not found for {split}. Run Cell 6 first.')
            return
        with open(label_file) as f:
            for line in f:
                name, label = line.strip().split(',')
                self.samples.append((name, int(label)))
        c = sum(1 for _, l in self.samples if l == 1)
        u = len(self.samples) - c
        print(f'{split:6s}: {len(self.samples)} patches | Changed={c} ({c/len(self.samples):.1%}) | Unchanged={u} ({u/len(self.samples):.1%})')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        name, label = self.samples[idx]
        ia = cv2.cvtColor(cv2.imread(os.path.join(self.root, 'A', name)), cv2.COLOR_BGR2RGB)
        ib = cv2.cvtColor(cv2.imread(os.path.join(self.root, 'B', name)), cv2.COLOR_BGR2RGB)
        out = self.transform(image=ia, image_b=ib)
        return out['image'], out['image_b'], torch.tensor(label, dtype=torch.long)


train_ds = ChangeDetectionDataset(PROCESSED_DIR, 'train')
val_ds   = ChangeDetectionDataset(PROCESSED_DIR, 'val')
test_ds  = ChangeDetectionDataset(PROCESSED_DIR, 'test')

# num_workers=0 for Windows compatibility (avoids multiprocessing issues)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print('\n✅ DataLoaders ready!')
ia, ib, labels = next(iter(train_loader))
print(f'Batch — img1: {ia.shape} | img2: {ib.shape} | labels: {labels.shape}')
print(f'Labels in batch: {labels.tolist()[:10]}...')

## CELL 8 — Model Architecture (Siamese Network)
Fixed: uses `weights=` API instead of deprecated `pretrained=` to remove warnings.
Fixed: ResNet50 `in_features` computed dynamically to avoid shape mismatch.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision.models import VGG16_Weights, ResNet50_Weights


class VGG16Encoder(nn.Module):
    def __init__(self, pretrained=True, freeze=True):
        super().__init__()
        weights = VGG16_Weights.IMAGENET1K_V1 if pretrained else None
        vgg = models.vgg16(weights=weights)
        f = vgg.features
        self.block1 = f[0:5]
        self.block2 = f[5:10]
        self.block3 = f[10:17]
        self.block4 = f[17:24]
        self.block5 = f[24:31]
        if freeze:
            for p in self.parameters(): p.requires_grad = False

    def forward(self, x):
        f1 = self.block1(x)
        f2 = self.block2(f1)
        f3 = self.block3(f2)
        f4 = self.block4(f3)
        f5 = self.block5(f4)
        return f1, f2, f3, f4, f5


class ResNet50Encoder(nn.Module):
    def __init__(self, pretrained=True, freeze=True):
        super().__init__()
        weights = ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        r = models.resnet50(weights=weights)
        self.l0 = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool)
        self.l1 = r.layer1
        self.l2 = r.layer2
        self.l3 = r.layer3
        self.l4 = r.layer4
        if freeze:
            for p in self.parameters(): p.requires_grad = False

    def forward(self, x):
        f0 = self.l0(x)
        f1 = self.l1(f0)
        f2 = self.l2(f1)
        f3 = self.l3(f2)
        f4 = self.l4(f3)
        return f0, f1, f2, f3, f4


class MultiLevelFusion(nn.Module):
    def __init__(self, use_blocks=[4, 5]):
        super().__init__()
        self.use_blocks = use_blocks

    def forward(self, ft1, ft2):
        sel1 = [ft1[b-1] for b in self.use_blocks]
        sel2 = [ft2[b-1] for b in self.use_blocks]
        target = sel1[0].shape[2:]
        r1, r2 = [], []
        for a, b in zip(sel1, sel2):
            if a.shape[2:] != target:
                a = F.interpolate(a, target, mode='bilinear', align_corners=False)
                b = F.interpolate(b, target, mode='bilinear', align_corners=False)
            r1.append(a); r2.append(b)
        c1 = torch.cat(r1, dim=1)
        c2 = torch.cat(r2, dim=1)
        return torch.cat([c1, c2], dim=1)


class DecisionNetwork(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),   # KEY FIX: pools spatial dims → removes shape mismatch
            nn.Flatten(),
            nn.Linear(in_features, 1024), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(1024, 256),          nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 2)
        )
    def forward(self, x):
        return self.net(x)


class SiameseChangeDetector(nn.Module):
    def __init__(self, backbone='vgg16', use_blocks=[4,5], pretrained=True, freeze=True):
        super().__init__()
        if backbone == 'vgg16':
            self.encoder = VGG16Encoder(pretrained, freeze)
            # 2 selected blocks x 512 channels x 2 sides = 2048
            in_feats = 512 * len(use_blocks) * 2
        else:
            self.encoder = ResNet50Encoder(pretrained, freeze)
            # blocks 4,5 → channels 1024, 2048; 2 sides
            block_channels = {1:256, 2:512, 3:1024, 4:2048, 5:2048}
            in_feats = sum(block_channels[b] for b in use_blocks) * 2
        self.fusion   = MultiLevelFusion(use_blocks)
        self.decision = DecisionNetwork(in_feats)

    def forward(self, x1, x2):
        f1    = self.encoder(x1)
        f2    = self.encoder(x2)
        fused = self.fusion(f1, f2)
        return self.decision(fused)

    def count_params(self):
        total = sum(p.numel() for p in self.parameters())
        train = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'Total params     : {total:,}')
        print(f'Trainable params : {train:,}')
        print(f'Frozen params    : {total-train:,}')


model = SiameseChangeDetector(
    backbone=BACKBONE, use_blocks=USE_BLOCKS,
    pretrained=PRETRAINED, freeze=FREEZE_BACKBONE
).to(DEVICE)

print('=== MODEL SUMMARY ===')
model.count_params()

# Test forward pass
x1 = torch.randn(2, 3, PATCH_SIZE, PATCH_SIZE).to(DEVICE)
x2 = torch.randn(2, 3, PATCH_SIZE, PATCH_SIZE).to(DEVICE)
with torch.no_grad():
    out = model(x1, x2)
print(f'\nForward pass OK! Output shape: {out.shape}  → [batch, 2 classes]')
print('✅ Model ready!')

## CELL 9 — Loss Functions

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        targets = targets.float()
        bce     = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt      = torch.exp(-bce)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * (1 - pt) ** self.gamma * bce).mean()


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        p = torch.sigmoid(logits).view(-1)
        t = targets.float().view(-1)
        inter = (p * t).sum()
        return 1 - (2 * inter + self.smooth) / (p.sum() + t.sum() + self.smooth)


class CrossEntropyBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
    def forward(self, logits, targets):
        return self.ce(logits, targets), torch.tensor(0.), torch.tensor(0.)


class HybridFocalDiceLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, focal_w=0.5, dice_w=0.5):
        super().__init__()
        self.focal   = FocalLoss(alpha, gamma)
        self.dice    = DiceLoss()
        self.focal_w = focal_w
        self.dice_w  = dice_w

    def forward(self, logits, targets):
        bin_logits = logits[:,1] - logits[:,0]
        fl = self.focal(bin_logits, targets.float())
        dl = self.dice(bin_logits, targets.float())
        return self.focal_w * fl + self.dice_w * dl, fl, dl


# Demo
print('=== LOSS FUNCTION COMPARISON ===')
ce_fn     = CrossEntropyBaseline()
hybrid_fn = HybridFocalDiceLoss(FOCAL_ALPHA, FOCAL_GAMMA, FOCAL_WEIGHT, DICE_WEIGHT)

logits = torch.tensor([[ 2.,-2.],[ 2.,-2.],[ 2.,-2.],[ 2.,-2.],
                        [ 2.,-2.],[ 2.,-2.],[ 2.,-2.],[-2., 2.]])
labels = torch.tensor([ 0,       0,       0,       0,       0,       0,       0,       1])

ce_loss, _, _      = ce_fn(logits, labels)
hy_loss, fl, dl    = hybrid_fn(logits, labels)

print(f'  Cross Entropy Loss : {ce_loss.item():.4f}  ← LOW, ignores missed change')
print(f'  Hybrid Focal+Dice  : {hy_loss.item():.4f}  ← Higher, penalizes missing change')
print(f'    Focal component  : {fl.item():.4f}')
print(f'    Dice  component  : {dl.item():.4f}')
print('\n✅ Loss functions ready!')

## CELL 10 — Evaluation Metrics

In [ ]:
import numpy as np
import torch
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, accuracy_score


class MetricsTracker:
    def __init__(self):
        self.reset()

    def reset(self):
        self.preds  = []
        self.labels = []

    def update(self, preds, labels):
        if isinstance(preds, torch.Tensor):
            if preds.dim() == 2: preds = preds.argmax(1)
            preds  = preds.cpu().numpy()
            labels = labels.cpu().numpy()
        self.preds.extend(preds.tolist())
        self.labels.extend(labels.tolist())

    def compute(self):
        p = np.array(self.preds)
        l = np.array(self.labels)
        if len(np.unique(l)) < 2:
            return {'accuracy':0,'precision':0,'recall':0,'f1':0,'iou':0,
                    'tp':0,'tn':0,'fp':0,'fn':0}
        cm = confusion_matrix(l, p)
        tn, fp, fn, tp = cm.ravel()
        return {
            'accuracy' : accuracy_score(l, p) * 100,
            'precision': precision_score(l, p, zero_division=0) * 100,
            'recall'   : recall_score(l, p, zero_division=0) * 100,
            'f1'       : f1_score(l, p, zero_division=0) * 100,
            'iou'      : tp / (tp + fp + fn + 1e-8) * 100,
            'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn)
        }

    def print_results(self, prefix=''):
        m = self.compute()
        print(f'  {prefix}Accuracy : {m["accuracy"]:6.2f}%')
        print(f'  {prefix}Precision: {m["precision"]:6.2f}%')
        print(f'  {prefix}Recall   : {m["recall"]:6.2f}%')
        print(f'  {prefix}F1 Score : {m["f1"]:6.2f}%')
        print(f'  {prefix}IoU      : {m["iou"]:6.2f}%')
        return m

print('✅ MetricsTracker ready!')

## CELL 11 — Trainer Class

In [ ]:
import torch, os
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


class Trainer:
    def __init__(self, model, train_loader, val_loader,
                 use_hybrid=True, exp_name='experiment'):
        self.model        = model.to(DEVICE)
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.exp_name     = exp_name
        self.ckpt_dir     = os.path.join(CHECKPOINT_DIR, exp_name)
        os.makedirs(self.ckpt_dir, exist_ok=True)

        if use_hybrid:
            self.criterion = HybridFocalDiceLoss(FOCAL_ALPHA, FOCAL_GAMMA, FOCAL_WEIGHT, DICE_WEIGHT)
            print(f'[{exp_name}] Loss: Hybrid Focal-Dice')
        else:
            self.criterion = CrossEntropyBaseline()
            print(f'[{exp_name}] Loss: Cross Entropy (baseline)')

        self.optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        self.scheduler = optim.lr_scheduler.StepLR(
            self.optimizer, LR_STEP_SIZE, LR_GAMMA)

        self.history = {k: [] for k in ['tl','vl','tf1','vf1','tacc','vacc','tiou','viou']}
        self.best_f1 = 0
        self.best_ep = 0
        self.train_m = MetricsTracker()
        self.val_m   = MetricsTracker()

    def _epoch(self, loader, train=True):
        self.model.train(train)
        mt  = self.train_m if train else self.val_m
        mt.reset()
        tot = 0
        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for x1, x2, y in loader:
                x1, x2, y = x1.to(DEVICE), x2.to(DEVICE), y.to(DEVICE)
                out        = self.model(x1, x2)
                loss, _, _ = self.criterion(out, y)
                if train:
                    self.optimizer.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    self.optimizer.step()
                tot += loss.item()
                mt.update(out.argmax(1), y)
        return tot / len(loader), mt.compute()

    def train(self, epochs=EPOCHS):
        print(f'\nTraining for {epochs} epochs...')
        print('Epoch | Loss(tr/val) | F1(tr/val) | IoU(tr/val) | Acc(tr/val)')
        print('-' * 72)
        for ep in range(1, epochs + 1):
            tl, tm = self._epoch(self.train_loader, train=True)
            vl, vm = self._epoch(self.val_loader,   train=False)
            self.scheduler.step()

            for k, v in [('tl',tl),('vl',vl),
                         ('tf1',tm['f1']),('vf1',vm['f1']),
                         ('tacc',tm['accuracy']),('vacc',vm['accuracy']),
                         ('tiou',tm['iou']),('viou',vm['iou'])]:
                self.history[k].append(v)

            is_best = vm['f1'] > self.best_f1
            if is_best:
                self.best_f1 = vm['f1']
                self.best_ep = ep
                torch.save({'epoch': ep, 'state': self.model.state_dict(), 'f1': vm['f1']},
                           os.path.join(self.ckpt_dir, 'best.pth'))

            star = '★' if is_best else ' '
            print(f'{star}Ep {ep:3d} | {tl:.4f}/{vl:.4f} | '
                  f'{tm["f1"]:5.1f}/{vm["f1"]:5.1f} | '
                  f'{tm["iou"]:5.1f}/{vm["iou"]:5.1f} | '
                  f'{tm["accuracy"]:5.1f}/{vm["accuracy"]:5.1f}')

        print(f'\n✅ Done! Best F1: {self.best_f1:.2f}% at epoch {self.best_ep}')
        self._plot()
        return self.history

    def _plot(self):
        fig, ax = plt.subplots(1, 3, figsize=(15, 4))
        e = range(1, len(self.history['tl']) + 1)
        ax[0].plot(e, self.history['tl'], 'b', label='Train')
        ax[0].plot(e, self.history['vl'], 'r', label='Val')
        ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid(True)
        ax[1].plot(e, self.history['tf1'], 'b', label='Train')
        ax[1].plot(e, self.history['vf1'], 'r', label='Val')
        ax[1].set_title('F1 Score (%)'); ax[1].legend(); ax[1].grid(True)
        ax[2].plot(e, self.history['tiou'], 'b', label='Train')
        ax[2].plot(e, self.history['viou'], 'r', label='Val')
        ax[2].set_title('IoU (%)'); ax[2].legend(); ax[2].grid(True)
        plt.suptitle(f'Training History — {self.exp_name}', fontsize=13)
        plt.tight_layout()
        save_path = os.path.join(RESULTS_DIR, f'history_{self.exp_name}.png')
        plt.savefig(save_path, dpi=120)
        plt.show()
        print(f'Plot saved to {save_path}')


print('✅ Trainer class ready!')

## CELL 12 — Experiment 1: Baseline (VGG16 + Cross Entropy)

In [ ]:
print('=== EXPERIMENT 1: Paper Baseline ===')
print('VGG16 + Cross Entropy Loss')
print()

model_baseline = SiameseChangeDetector(
    backbone='vgg16', use_blocks=[4,5],
    pretrained=True, freeze=True
).to(DEVICE)

trainer_baseline = Trainer(
    model_baseline, train_loader, val_loader,
    use_hybrid=False, exp_name='baseline_CE'
)

history_baseline = trainer_baseline.train(epochs=EPOCHS)

## CELL 13 — Experiment 2: Improved (VGG16 + Hybrid Loss)

In [ ]:
print('=== EXPERIMENT 2: Improved ===')
print('VGG16 + Hybrid Focal-Dice Loss')
print()

model_improved = SiameseChangeDetector(
    backbone='vgg16', use_blocks=[4,5],
    pretrained=True, freeze=True
).to(DEVICE)

trainer_improved = Trainer(
    model_improved, train_loader, val_loader,
    use_hybrid=True, exp_name='improved_FocalDice'
)

history_improved = trainer_improved.train(epochs=EPOCHS)

## CELL 14 — Experiment 3: Best (ResNet50 + Hybrid Loss)

In [ ]:
print('=== EXPERIMENT 3: Best Model ===')
print('ResNet50 + Hybrid Focal-Dice Loss')
print()

model_best = SiameseChangeDetector(
    backbone='resnet50', use_blocks=[4,5],
    pretrained=True, freeze=True
).to(DEVICE)

trainer_best = Trainer(
    model_best, train_loader, val_loader,
    use_hybrid=True, exp_name='best_ResNet50'
)

history_best = trainer_best.train(epochs=EPOCHS)

## CELL 15 — Final Evaluation on Test Set

In [ ]:
import torch

def evaluate_on_test(model, test_loader, name):
    model.eval()
    mt = MetricsTracker()
    with torch.no_grad():
        for x1, x2, y in test_loader:
            out = model(x1.to(DEVICE), x2.to(DEVICE))
            mt.update(out.argmax(1), y)
    print(f'\n=== {name} — TEST RESULTS ===')
    return mt.print_results()


def load_best(model, exp_name):
    path = os.path.join(CHECKPOINT_DIR, exp_name, 'best.pth')
    if os.path.exists(path):
        ck = torch.load(path, map_location=DEVICE)
        model.load_state_dict(ck['state'])
        print(f'Loaded best checkpoint for {exp_name} (F1={ck["f1"]:.2f}%)')
    return model


model_baseline = load_best(model_baseline, 'baseline_CE')
model_improved = load_best(model_improved, 'improved_FocalDice')
model_best     = load_best(model_best,     'best_ResNet50')

r1 = evaluate_on_test(model_baseline, test_loader, 'Baseline (VGG16+CE)')
r2 = evaluate_on_test(model_improved, test_loader, 'Improved (VGG16+Focal-Dice)')
r3 = evaluate_on_test(model_best,     test_loader, 'Best (ResNet50+Focal-Dice)')

print('\n' + '='*65)
print('          FINAL COMPARISON TABLE')
print('='*65)
print(f'{"Model":<30} {"Acc":>6} {"F1":>6} {"IoU":>6} {"Prec":>6} {"Rec":>6}')
print('-'*65)
for name, r in [
    ('Baseline (VGG16+CE)',        r1),
    ('Improved (VGG16+FocalDice)', r2),
    ('Best (ResNet50+FocalDice)',  r3)
]:
    print(f'{name:<30} {r["accuracy"]:5.1f}% {r["f1"]:5.1f}% {r["iou"]:5.1f}% {r["precision"]:5.1f}% {r["recall"]:5.1f}%')
print('='*65)

## CELL 16 — Compare Training Curves

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Experiment Comparison — Validation Metrics', fontsize=14)

styles = [
    (history_baseline, 'Baseline (VGG16+CE)',        'gray',   '--'),
    (history_improved, 'Improved (VGG16+FocalDice)', '#e67e22', '-.'),
    (history_best,     'Best (ResNet50+FocalDice)',  '#2980b9', '-'),
]

for hist, label, color, ls in styles:
    e = range(1, len(hist['vl']) + 1)
    axes[0].plot(e, hist['vl'],   color=color, ls=ls, label=label, lw=2)
    axes[1].plot(e, hist['vf1'],  color=color, ls=ls, label=label, lw=2)
    axes[2].plot(e, hist['viou'], color=color, ls=ls, label=label, lw=2)

axes[0].set_title('Val Loss');    axes[0].grid(True); axes[0].legend(fontsize=8)
axes[1].set_title('Val F1 (%)'); axes[1].grid(True); axes[1].legend(fontsize=8)
axes[2].set_title('Val IoU (%)');axes[2].grid(True); axes[2].legend(fontsize=8)

plt.tight_layout()
save_path = os.path.join(RESULTS_DIR, 'experiment_comparison.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Comparison plot saved to {save_path}')

## CELL 17 — Save Results Summary

In [ ]:
import json, os

results_summary = {
    'Baseline_VGG16_CE'        : r1,
    'Improved_VGG16_FocalDice' : r2,
    'Best_ResNet50_FocalDice'  : r3,
}

with open(os.path.join(RESULTS_DIR, 'results.json'), 'w') as f:
    json.dump(results_summary, f, indent=2)

print('=== RESULTS SAVED ===')
print(f'Location: {RESULTS_DIR}')
print()
print('Files generated:')
for fname in os.listdir(RESULTS_DIR):
    fpath = os.path.join(RESULTS_DIR, fname)
    size  = os.path.getsize(fpath)
    print(f'  {fname:45s} ({size//1024} KB)')

## CELL 18 — Single Image Prediction (Demo)
Provide paths to two local images (before & after) to test prediction.

In [ ]:
import cv2, torch, numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


def predict_pair(model, img_before_path, img_after_path):
    transform = A.Compose([
        A.Resize(PATCH_SIZE, PATCH_SIZE),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2()
    ], additional_targets={'image_b': 'image'})

    ia  = cv2.cvtColor(cv2.imread(img_before_path), cv2.COLOR_BGR2RGB)
    ib  = cv2.cvtColor(cv2.imread(img_after_path),  cv2.COLOR_BGR2RGB)
    out = transform(image=ia, image_b=ib)
    x1  = out['image'].unsqueeze(0).to(DEVICE)
    x2  = out['image_b'].unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(x1, x2)
        probs  = torch.softmax(logits, 1)[0]
        pred   = logits.argmax(1).item()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(ia); axes[0].set_title('Before'); axes[0].axis('off')
    axes[1].imshow(ib); axes[1].set_title('After');  axes[1].axis('off')
    plt.suptitle(
        f'Prediction: {"CHANGED" if pred==1 else "NO CHANGE"} (confidence: {probs[1].item():.1%})',
        fontsize=13, color='red' if pred == 1 else 'green'
    )
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'single_prediction.png'), dpi=120)
    plt.show()
    return pred, probs[1].item()


# ── Change these paths to your own images ──
before_path = r'C:\path\to\before_image.png'
after_path  = r'C:\path\to\after_image.png'

if os.path.exists(before_path) and os.path.exists(after_path):
    pred, conf = predict_pair(model_best, before_path, after_path)
    print(f'Result     : {"CHANGED" if pred==1 else "NO CHANGE"}')
    print(f'Confidence : {conf:.1%}')
else:
    print('Update before_path and after_path above with real image paths to use this cell.')